# Testing Context Extraction with AST

In [74]:
# imports
import ast
from typing import Set, Dict, List

In [75]:
# example to understand AST
code = """
# Globale Variablen
glob_a = 10
x: int = 5

# Funktionen
def func1():
    return glob_a + helper1()

def helper1():
    return 5
"""

## AST Representation

In [76]:
# AST-tree hierarchical representation
tree = ast.parse(code)
print(ast.dump(tree, indent=4))

Module(
    body=[
        Assign(
            targets=[
                Name(id='glob_a', ctx=Store())],
            value=Constant(value=10)),
        AnnAssign(
            target=Name(id='x', ctx=Store()),
            annotation=Name(id='int', ctx=Load()),
            value=Constant(value=5),
            simple=1),
        FunctionDef(
            name='func1',
            args=arguments(),
            body=[
                Return(
                    value=BinOp(
                        left=Name(id='glob_a', ctx=Load()),
                        op=Add(),
                        right=Call(
                            func=Name(id='helper1', ctx=Load()))))]),
        FunctionDef(
            name='helper1',
            args=arguments(),
            body=[
                Return(
                    value=Constant(value=5))])])


In [77]:
# go through all nodes with BFS
for node in ast.walk(tree):
        print(node)

Module(body=[Assign(targets=[Name(id='glob_a', ctx=Store(...))], value=Constant(value=10, kind=None), type_comment=None), ..., FunctionDef(name='helper1', args=arguments(posonlyargs=[], args=[], vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]), body=[Return(value=Constant(...))], decorator_list=[], returns=None, type_comment=None, type_params=[])], type_ignores=[])
Assign(targets=[Name(id='glob_a', ctx=Store())], value=Constant(value=10, kind=None), type_comment=None)
AnnAssign(target=Name(id='x', ctx=Store()), annotation=Name(id='int', ctx=Load()), value=Constant(value=5, kind=None), simple=1)
FunctionDef(name='func1', args=arguments(posonlyargs=[], args=[], vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]), body=[Return(value=BinOp(left=Name(...), op=Add(...), right=Call(...)))], decorator_list=[], returns=None, type_comment=None, type_params=[])
FunctionDef(name='helper1', args=arguments(posonlyargs=[], args=[], vararg=None, kwonlyargs=[], kw_d

In [78]:
# only go through nodes on body-level
for node in tree.body:
        print(node)

Assign(targets=[Name(id='glob_a', ctx=Store())], value=Constant(value=10, kind=None), type_comment=None)
AnnAssign(target=Name(id='x', ctx=Store()), annotation=Name(id='int', ctx=Load()), value=Constant(value=5, kind=None), simple=1)
FunctionDef(name='func1', args=arguments(posonlyargs=[], args=[], vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]), body=[Return(value=BinOp(left=Name(...), op=Add(...), right=Call(...)))], decorator_list=[], returns=None, type_comment=None, type_params=[])
FunctionDef(name='helper1', args=arguments(posonlyargs=[], args=[], vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]), body=[Return(value=Constant(value=5, kind=None))], decorator_list=[], returns=None, type_comment=None, type_params=[])


## Global Variables - Extraction

In [79]:
# global assignments
for node in tree.body:

    if isinstance(node, ast.Assign):
        print(node)
    
    if isinstance(node, ast.AnnAssign):
        print(node)

Assign(targets=[Name(id='glob_a', ctx=Store())], value=Constant(value=10, kind=None), type_comment=None)
AnnAssign(target=Name(id='x', ctx=Store()), annotation=Name(id='int', ctx=Load()), value=Constant(value=5, kind=None), simple=1)


In [80]:
# global assignment-targets (left side of an assignment)
for node in tree.body:

    if isinstance(node, ast.Assign):
        for target in node.targets:
            print(target)
    
    if isinstance(node, ast.AnnAssign):
        print(node.target)

Name(id='glob_a', ctx=Store())
Name(id='x', ctx=Store())


In [81]:
# global variables
for node in tree.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name):
                print(target.id)

    if isinstance(node, ast.AnnAssign):
            if isinstance(node.target, ast.Name):
                print(node.target.id)

glob_a
x


## Global Functions - Extraction

In [82]:
# Globale erkennen und in Dict speichern
functions = {}
for node in tree.body:
    if isinstance(node, ast.FunctionDef):
        functions[node.name] = node

print(functions)


{'func1': FunctionDef(name='func1', args=arguments(posonlyargs=[], args=[], vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]), body=[Return(value=BinOp(left=Name(...), op=Add(...), right=Call(...)))], decorator_list=[], returns=None, type_comment=None, type_params=[]), 'helper1': FunctionDef(name='helper1', args=arguments(posonlyargs=[], args=[], vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]), body=[Return(value=Constant(value=5, kind=None))], decorator_list=[], returns=None, type_comment=None, type_params=[])}


## Example with nested function calls

In [83]:
fileCode = """
# Globale Variablen
glob_a = 10
glob_b = 20
glob_c = 30
glob_d = [1, 2, 3]
glob_e = "Hallo"
glob_f = {"x": 1}
x: int = 5
y = 10

# Funktionen
def func1():
    return glob_a + helper1() + x

def helper1():
    return glob_b + helper2()

def helper2():
    return glob_c + helper3()

def helper3():
    return y + helper4()

def helper4():
    return 42

def dummy1():
    return glob_c

def dummy2():
    test = 5;
    return glob_d
"""

# function to analyze
functionToAnalyze = "func1"

## Finale Klasse für AST Analyse

In [84]:
# AST parser class for extracting references to functions and global variables
class FunctionAnalyzer(ast.NodeVisitor):
    def __init__(self, function_name: str, tree: ast.AST):
        self.function_name = function_name
        self.tree = tree
        self.called_functions: Set[str] = set()
        self.used_globals: Set[str] = set()
        self.global_vars = self._get_global_vars()
        self.function_defs = self._get_function_defs()

    def _get_global_vars(self) -> Dict[str, ast.AST]:
        global_variables  = {}

        for node in self.tree.body:
            # normal assignment (e.g. x = 5 | can have multiple targets e.g. a = b = c = 5)
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name):
                        global_variables[target.id] = node

            # annotated assignment (x: int = 5 | exactly one target)
            if isinstance(node, ast.AnnAssign):
                if isinstance(node.target, ast.Name):
                    global_variables[node.target.id] = node

        return global_variables

    def _get_function_defs(self) -> Dict[str, ast.FunctionDef]:
        functions = {}
        for node in self.tree.body:
            if isinstance(node, ast.FunctionDef):
                functions[node.name] = node
        return functions
    
    # finds all function & global variables
    def analyze_function(self, fn_name: str, visited: Set[str] = None) -> Dict[str, Set[str]]:
        if visited is None:
            visited = set()
        if fn_name in visited:
            return {"functions": set(), "globals": set()}

        visited.add(fn_name)

        if fn_name not in self.function_defs:
            return {"functions": set(), "globals": set()}

        node = self.function_defs[fn_name]
        local_called_functions = set()
        local_used_globals = set()

        for child in ast.walk(node):
            # function calls
            if isinstance(child, ast.Call) and isinstance(child.func, ast.Name) and child.func.id in self.function_defs:
                local_called_functions.add(child.func.id)

            # usage of global variables
            if isinstance(child, ast.Name) and child.id in self.global_vars:
                local_used_globals.add(child.id)

        # recursively investigate further functional dependencies
        all_functions = set(local_called_functions)
        all_globals = set(local_used_globals)

        for called_fn in local_called_functions:
            deeper = self.analyze_function(called_fn, visited)
            all_functions.update(deeper["functions"])
            all_globals.update(deeper["globals"])

        return {"functions": all_functions, "globals": all_globals}
    
    # creates a context string containing relevant global variables and relevant helper functions
    def create_context(self, analysis_result: Dict[str, Set[str]], fn_name: str, source_code: str) -> str:
        relevant_functions = analysis_result["functions"]
        relevant_globals = analysis_result["globals"]

        context_parts = []

        # global variables (sort as in code)
        sorted_globals = sorted(
            ((var, node) for var, node in self.global_vars.items() if var in relevant_globals),
            key=lambda x: x[1].lineno
        )

        for str, node in sorted_globals:
            context_parts.append(
                ast.get_source_segment(source_code, node)
            )

        # helper functions (sort by minimum dependencies)
        visited: Set[str] = set()
        ordered_functions: List[str] = []

        def dfs(fn: str):
            if fn in visited or fn == fn_name:
                return
            visited.add(fn)
 
            fn_node = self.function_defs.get(fn)
            if fn_node is None:
                return
       
            called = set()
            for child in ast.walk(fn_node):
                if isinstance(child, ast.Call) and isinstance(child.func, ast.Name):
                    called_fn = child.func.id
                    if called_fn in self.function_defs:
                        called.add(called_fn)
    
            for c in called:
                dfs(c)
  
            ordered_functions.append(fn)

        for fn in relevant_functions:
            dfs(fn)

        for fn in ordered_functions:
            fn_node = self.function_defs.get(fn)
            if fn_node:
                context_parts.append(ast.get_source_segment(source_code, fn_node))

        return "\n\n".join(context_parts)
    
    def get_sourcecode_for_function(self, fn_name: str, fileCode : str) -> str:
        sourcecode = ast.get_source_segment(fileCode, self.function_defs[fn_name])
        return sourcecode


In [85]:
# Main Program
if __name__ == "__main__":
    # create AST of whole program
    tree = ast.parse(fileCode)

    analyzer = FunctionAnalyzer(functionToAnalyze, tree)
    result = analyzer.analyze_function(functionToAnalyze)

    # extract source code of the main function
    main_function_source = analyzer.get_sourcecode_for_function(functionToAnalyze, fileCode)

    # create context (global variables + helper functions)
    context_string = analyzer.create_context(analysis_result=result, fn_name=functionToAnalyze, source_code=fileCode)

    result = context_string + "\n" + "\n" + main_function_source

    # print("--- MAIN FUNCTION ---")
    # print(main_function_source)

    # print("\n--- CONTEXT ---")
    # print(context_string)

    print("\n--- RESULT ---")
    print(result)



--- RESULT ---
glob_a = 10

glob_b = 20

glob_c = 30

x: int = 5

y = 10

def helper4():
    return 42

def helper3():
    return y + helper4()

def helper2():
    return glob_c + helper3()

def helper1():
    return glob_b + helper2()

def func1():
    return glob_a + helper1() + x
